In [1]:
# Imports and Load Interim Data
import pandas as pd
import numpy as np
import re

# Load our filtered data from Notebook 1
df_books = pd.read_csv('data/interim_books.csv')
df_nyt = pd.read_csv('data/bestsellers.csv')

print(f"Loaded {len(df_books):,} filtered books.")

Loaded 458,516 filtered books.


Matching "The Great Gatsby" to "Great Gatsby, The" requires cleaning. This function standardizes titles to ensure the merge works correctly.

In [2]:
# Title Standardization for Matching
def clean_title(title):
    if not isinstance(title, str): return ""
    title = title.lower()
    # Remove subtitles (after colons/dashes) and special characters
    title = re.sub(r'[:\-\(\)].*', '', title)
    title = re.sub(r'[^a-z0-9\s]', '', title)
    return title.strip()

df_books['title_clean'] = df_books['title'].apply(clean_title)
df_nyt['title_clean'] = df_nyt['title'].apply(clean_title)

We perform a "Left Join" to see which of our Goodreads books appear on the NYT list. If they match, bestseller = 1, otherwise 0.

In [3]:
# Creating the Target Label (Bestseller)
# Create a unique list of NYT titles
nyt_titles = df_nyt['title_clean'].unique()

# Assign the bestseller label
df_books['bestseller'] = df_books['title_clean'].isin(nyt_titles).astype(int)

print(f"Bestseller Distribution:\n{df_books['bestseller'].value_counts()}")
print(f"Bestseller Rate: {(df_books['bestseller'].mean()*100):.4f}%")

Bestseller Distribution:
bestseller
0    430446
1     28070
Name: count, dtype: int64
Bestseller Rate: 6.1219%


The reviews file is too large for standard loading. We will load only the review_text for books we have already filtered into df_books.

In [4]:
# Preparing Reviews for Sentiment Analysis
reviews_path = 'data/goodreads_reviews_dedup.json.gz'
relevant_book_ids = set(df_books['book_id'].unique())

# We will read reviews in chunks and only keep those for our filtered books
review_chunks = []
print("Filtering reviews for relevant books...")

reader = pd.read_json(reviews_path, lines=True, compression='infer', chunksize=100000)

for i, chunk in enumerate(reader):
    # Only keep reviews for books that passed our 2010-2019/English filter
    chunk = chunk[chunk['book_id'].isin(relevant_book_ids)]
    
    # Keep only columns needed for sentiment
    review_chunks.append(chunk[['book_id', 'review_text', 'rating']])
    
    if (i + 1) % 10 == 0:
        print(f"Processed {(i + 1) * 100000:,} reviews...")

df_reviews = pd.concat(review_chunks, ignore_index=True)
print(f"Final Review Count: {len(df_reviews):,}")

Filtering reviews for relevant books...
Processed 1,000,000 reviews...
Processed 2,000,000 reviews...
Processed 3,000,000 reviews...
Processed 4,000,000 reviews...
Processed 5,000,000 reviews...
Processed 6,000,000 reviews...
Processed 7,000,000 reviews...
Processed 8,000,000 reviews...
Processed 9,000,000 reviews...
Processed 10,000,000 reviews...
Processed 11,000,000 reviews...
Processed 12,000,000 reviews...
Processed 13,000,000 reviews...
Processed 14,000,000 reviews...
Processed 15,000,000 reviews...
Final Review Count: 6,626,356


In [11]:
# Save Merged Checkpoint
# Save the books with their new 'bestseller' labels
df_books.to_csv('data/books_labeled.csv', index=False)

# Save the filtered reviews for the Sentiment Analysis notebook
df_reviews.to_csv('data/reviews_filtered.csv', index=False)

print("Notebook 2 complete. Ready for Sentiment Analysis.")

Notebook 2 complete. Ready for Sentiment Analysis.


In [12]:
import pandas as pd
temp = pd.read_csv('data/interim_books.csv', nrows=5)
print(temp.columns.tolist())

['book_id', 'title', 'authors', 'publisher', 'publication_year', 'average_rating', 'ratings_count', 'text_reviews_count', 'description']


We use Leave-One-Out (LOO) encoding for Author Reputation.
This ensures that for any book, the 'author_reputation' feature only counts 
the author's OTHER bestsellers, preventing the target variable from leaking into the training set.

In [17]:
import pandas as pd
import re

## --- FEATURE ENGINEERING: PREVENTING DATA LEAKAGE (SAFE VERSION) ---

# Check if we already processed this to avoid KeyError on re-runs
if 'author_reputation' in df_books.columns:
    print("Author Reputation already exists. Skipping extraction.")
else:
    # 1. Extract Author ID (Handles strings, lists, or dictionaries)
    def clean_author(x):
        s = str(x)
        match = re.search(r"'author_id':\s*'(\d+)'", s)
        if match:
            return match.group(1)
        # If it's just a raw number in the string
        digits = re.sub(r"\D", "", s)
        return digits if digits else 'unknown'

    print("Extracting author IDs...")
    # Ensure 'authors' exists before applying
    if 'authors' in df_books.columns:
        df_books['author_id'] = df_books['authors'].apply(clean_author)
        
        # --- HEALTH CHECK ---
        unique_authors = df_books['author_id'].nunique()
        print(f"Total books: {len(df_books)}")
        print(f"Unique authors found: {unique_authors}")

        if unique_authors < 100:
            print("ERROR: Data still looks grouped incorrectly. Check 'authors' column content.")
        else:
            # 2. Calculate total bestsellers per author
            author_counts = df_books.groupby('author_id')['bestseller'].sum().reset_index()
            author_counts.columns = ['author_id', 'total_author_bestsellers']

            # 3. Merge and Apply LOO
            df_books = df_books.merge(author_counts, on='author_id', how='left')
            df_books['author_reputation'] = df_books.apply(
                lambda x: x['total_author_bestsellers'] - 1 if x['bestseller'] == 1 else x['total_author_bestsellers'], 
                axis=1
            )

            # 4. Clean up
            df_books.drop(columns=['total_author_bestsellers', 'authors'], inplace=True)
            print("Success! Author Reputation engineered correctly.")
            print(df_books[['title', 'author_id', 'author_reputation']].head(10))
    else:
        print("ERROR: 'authors' column not found. Please re-run the notebook from the data loading step.")

Author Reputation already exists. Skipping extraction.


In [18]:
# Verify the data is healthy
print(df_books[['title', 'author_id', 'author_reputation']].head(10))
print("\nUnique values in reputation:", df_books['author_reputation'].unique())

                                               title author_id  \
0                                   Glimmering Light   unknown   
1       The 30s (Fantastic Films of the Decades, #2)   unknown   
2                         Untold Secrets: Fire & Ice   unknown   
3          Understand God's Word - Walk in the Truth   unknown   
4                              A Murder is Announced   unknown   
5  Captain America: Winter Soldier (The Ultimate ...   unknown   
6                                              Smoke   unknown   
7            Healer's Touch (Hearts And Thrones, #4)   unknown   
8                     Nemesis (Southern Comfort, #4)   unknown   
9                                  End-Time Gangster   unknown   

   author_reputation  
0              28070  
1              28070  
2              28070  
3              28070  
4              28070  
5              28069  
6              28070  
7              28070  
8              28069  
9              28070  

Unique values in re

In [19]:
## --- DATA EXPORT ---
# Saving the labeled dataset with the new Author Reputation feature 
# for use in the sentiment merging notebook.

df_books.to_csv('data/books_labeled.csv', index=False)
print("Notebook 2 complete: Labeled data saved to data/books_labeled.csv")

Notebook 2 complete: Labeled data saved to data/books_labeled.csv
